# Gurbani CTC Batch Alignment — v2 (Mac M1 / Colab / Kaggle)

Aligns Sehaj Paath audio (Bhagat Jaswant Singh Ji) to Gurbani text with a
Punjabi wav2vec2 CTC forced aligner. Outputs **per-word JSON + line SRT**
per Ang, with parallel workers.

**Version v2** — re-upload only when the pipeline code changes;
otherwise just edit the Ang range in the config cell and re-run the same upload.

**On a Mac (M1/M2/M3):** open this notebook in Jupyter/VS Code. It uses MPS
(Apple GPU) when available, else CPU. 2 workers is a good default.

**On Google Colab:** upload this notebook, then Runtime → Change runtime type
→ **T4 GPU**. The GPU is ~5–10× faster per Ang than a laptop CPU. 2 workers
share the GPU fine.

**On Kaggle:** Code → New Notebook → ⋮ menu → Upload notebook (or File →
Import). Then in Settings (right sidebar) set **Accelerator → GPU** and
**Internet → On**, then Run All. (Kaggle requires phone verification once
for GPU + internet access.)

**Steps:** set your Ang range in the config cell, then Run All.
Fetches text from readgurbani.com and audio from media.gursevak.com
automatically. Progress shows per Ang; a ZIP of all JSON+SRT is built at the
end for download.

**Checkpoints:** every 35 successful Angs a cumulative
`gurbani_batch_checkpoint_N.zip` is written to disk — if the run dies
partway, download the latest checkpoint instead of starting over.


In [ ]:
import importlib.util, sys, subprocess

def need(mod, pip):
    if importlib.util.find_spec(mod) is None:
        print(f"installing {pip} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip])
    else:
        print(f"{mod}: ok")

need("torch", "torch")                      # Colab: preinstalled (CUDA). Mac: MPS/CPU build
need("transformers", "transformers")
need("librosa", "librosa")
need("ctc_forced_aligner", "ctc-forced-aligner")
need("tqdm", "tqdm")

In [ ]:
%%writefile gurbani_worker.py
#!/usr/bin/env python3
"""
Portable Gurbani CTC forced-alignment worker.

Self-contained: no dependency on timestamper.py. Runs on
  - Mac (Apple Silicon MPS, or CPU),
  - Google Colab (CUDA GPU, or CPU),
  - any Linux CPU box.

Pipeline per Ang:
  1. fetch text (readgurbani.com TSV -> one line per row) and audio
     (media.gursevak.com Sehaj Paath MP3)
  2. manandey/wav2vec2-large-xlsr-punjabi -> frame posteriors (20ms,
     chunked to bound memory, cached as .npz keyed by sample count)
  3. Viterbi forced alignment of the canonical char sequence
     (nukta precomposed; ੴ -> "ਇਕ ਓਅੰਕਾਰ"; digit markers unspoken)
  4. word timestamps -> JSON (same schema as the desktop pipeline)
     + line-level SRT.

Used two ways:
  - imported by the batch notebook (ProcessPoolExecutor workers), or
  - python gurbani_worker.py 12 20 --out my_output
"""

import argparse
import csv
import io
import json
import os
import re
import sys
import time
import unicodedata
import urllib.request
import urllib.error

import numpy as np

MODEL_ID = "manandey/wav2vec2-large-xlsr-punjabi"
FRAME_S = 0.02
BLANK_ID = 62
SEP_ID = 22  # '|' word separator

UA = {"User-Agent": "Mozilla/5.0 (compatible; gurbani-timestamper)"}
TEXT_URL = "https://readgurbani.com/data/sggs/gurbani/{ang}.csv"
AUDIO_URL = ("https://media.gursevak.com/media/Sehaj_Paath_Pagewise/"
             "Sehaj_Paath_16kbps/{ang:04d}.mp3")

# ------------------------------------------------------------ normalization

PUNCT_RE = re.compile(r"[।॥.,;:!?\-–—\"'()\[\]…]")
WS_RE = re.compile(r"\s+")
GURMUKHI_DIGITS = str.maketrans("੦੧੨੩੪੫੬੭੮੯", "0123456789")
START_PAD = 0.15
END_PAD = 0.30


def norm(t: str) -> str:
    t = unicodedata.normalize("NFC", t)
    t = t.translate(GURMUKHI_DIGITS)
    t = PUNCT_RE.sub("", t)
    return WS_RE.sub(" ", t).strip()


def load_lines(path):
    with open(path, encoding="utf-8") as f:
        return [ln.strip() for ln in f if ln.strip()]


# ------------------------------------------------------------ line builders

def build_cues(lines, ref_line, hyp_words, pairs, audio_dur):
    """ref_line: norm-word index -> line index. Returns (cues, hits)."""
    hits = {}
    for i, j in pairs:
        hits.setdefault(ref_line[i], []).append(j)

    bounds = {}
    for li, js in hits.items():
        s = hyp_words[min(js)]["start"] - START_PAD
        e = hyp_words[max(js)]["end"] + END_PAD
        bounds[li] = (max(0.0, s), e)

    anchored = sorted(bounds)
    cues = []
    for li in range(len(lines)):
        if li in bounds:
            cues.append(bounds[li])
            continue
        prev = max([a for a in anchored if a < li], default=None)
        nxt = min([a for a in anchored if a > li], default=None)
        if prev is None and nxt is None:
            cues.append((0.0, audio_dur))
        elif prev is None:
            cues.append((0.0, bounds[nxt][0]))
        elif nxt is None:
            cues.append((bounds[prev][1], audio_dur))
        else:
            lo, hi = bounds[prev][1], bounds[nxt][0]
            wc = [len(norm(lines[k]).split()) for k in range(prev + 1, nxt)]
            total = sum(wc) or 1
            before = sum(wc[: li - prev - 1])
            frac0 = before / total
            frac1 = (before + wc[li - prev - 1]) / total
            cues.append((lo + frac0 * (hi - lo), lo + frac1 * (hi - lo)))

    fixed = []
    for s, e in cues:
        s = max(0.0, min(s, audio_dur))
        e = max(0.0, min(e, audio_dur))
        if e <= s:
            e = min(s + 0.5, audio_dur)
        fixed.append([s, e])
    for k in range(1, len(fixed)):
        if fixed[k][0] < fixed[k - 1][1]:
            mid = (fixed[k][0] + fixed[k - 1][1]) / 2
            fixed[k - 1][1] = mid
            fixed[k][0] = mid
    return fixed, hits


def build_word_timings(lines, ref, hyp_words, pairs, line_cues):
    """ref: list of (orig_tok, norm_w, line_idx)."""
    match = {}
    for i, j in pairs:
        hw = hyp_words[j]
        match[i] = (hw["start"], hw["end"])

    by_line = {}
    for i, (_o, _n, li) in enumerate(ref):
        by_line.setdefault(li, []).append(i)

    out = []
    for li, _line in enumerate(lines):
        idxs = by_line.get(li, [])
        n = len(idxs)
        cue_s, cue_e = line_cues[li]
        spans = [None] * n
        for pos, i in enumerate(idxs):
            if i in match:
                spans[pos] = [match[i][0], match[i][1]]
        bounds = [(-1, cue_s, cue_s)]
        for pos, se in enumerate(spans):
            if se is not None:
                bounds.append((pos, se[0], se[1]))
        bounds.append((n, cue_e, cue_e))
        for b in range(len(bounds) - 1):
            p0, _, e0 = bounds[b]
            p1, s1, _ = bounds[b + 1]
            lo, hi = e0, max(s1, e0)
            between = list(range(p0 + 1, p1))
            if not between:
                continue
            weights = [max(len(ref[idxs[p]][1]), 1) for p in between]
            tot = sum(weights)
            t = lo
            for p, wgt in zip(between, weights):
                d = (hi - lo) * wgt / tot
                spans[p] = [t, t + d]
                t += d
        for p in range(n):
            s, e = spans[p]
            s = max(s, cue_s)
            e = min(e, cue_e)
            if e <= s:
                e = s + 0.05
            spans[p] = [round(s, 2), round(e, 2)]
        for p in range(1, n):
            if spans[p][0] < spans[p - 1][1]:
                spans[p][0] = spans[p - 1][1]
                if spans[p][1] <= spans[p][0]:
                    spans[p][1] = round(spans[p][0] + 0.05, 2)
        out.append([
            {"word": ref[idxs[p]][0], "start": spans[p][0], "end": spans[p][1]}
            for p in range(n)
        ])
    return out


# ------------------------------------------------------------ fetching

def http_get(url, timeout=60, retries=4):
    last = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers=UA)
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read()
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            last = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"GET failed after {retries} tries: {url} ({last})")


def fetch_ang(ang, text_dir="text", audio_dir="audio"):
    if not 1 <= ang <= 1430:
        raise ValueError(f"ang out of range: {ang}")
    os.makedirs(text_dir, exist_ok=True)
    os.makedirs(audio_dir, exist_ok=True)

    txt_path = os.path.join(text_dir, f"ang{ang}.txt")
    if not (os.path.exists(txt_path) and os.path.getsize(txt_path)):
        raw = http_get(TEXT_URL.format(ang=ang)).decode("utf-8")
        if not raw.lstrip().startswith("id\t"):
            raise RuntimeError(f"ang {ang}: unexpected text response")
        rows = list(csv.DictReader(io.StringIO(raw), delimiter="\t"))
        with open(txt_path, "w", encoding="utf-8") as f:
            for r in rows:
                line = (r.get("gurmukhi") or "").strip()
                if line:
                    f.write(line + "\n")

    audio_path = os.path.join(audio_dir, f"ang{ang}.mp3")
    if not (os.path.exists(audio_path) and os.path.getsize(audio_path)):
        data = http_get(AUDIO_URL.format(ang=ang))
        with open(audio_path, "wb") as f:
            f.write(data)
    return txt_path, audio_path


# ------------------------------------------------------------ alignment

DIGIT_RE = re.compile(r"[0-9੦-੯]+$")
NUKTA = [("\u0a17\u0a3c", "\u0a5a"), ("\u0a38\u0a3c", "\u0a36"),
         ("\u0a1c\u0a3c", "\u0a5b"), ("\u0a16\u0a3c", "\u0a59"),
         ("\u0a2b\u0a3c", "\u0a5e"), ("\u0a32\u0a3c", "\u0a33")]
IK_OANKAR_EXPANSION = ["ਇਕ", "ਓਅੰਕਾਰ"]
MAX_SKIP = 4
# chars missing from the wav2vec2 Punjabi model vocab -> closest available
VOCAB_FIXUPS = [
    ("ਞ", "ਨ"),   # U+0A1E NYA -> NA (closest nasal)
    ("ਙ", "ਨ"),   # U+0A19 NGA -> NA (closest nasal)
    ("ਃ", "ਹ"),   # U+0A03 VISARGA -> HA (breathy release)
    ("ੵ", "ਯ"),   # U+0A75 YAKASH -> YA
    ("ੑ", ""),    # U+0A51 UDAAT (tone mark, no consonantal value) -> drop
    ("ੲ", "ਇ"),   # U+0A72 IRI: vowel base pronounced "i" (e.g. ਪਾੲਂੀ "ਪਾਈਂ")
    # subscript/superscript digits: unspoken footnote markers -> drop
    ("₀", ""), ("₁", ""), ("₂", ""), ("₃", ""), ("₄", ""),
    ("₅", ""), ("₆", ""), ("₇", ""), ("₈", ""), ("₉", ""),
    ("⁰", ""), ("¹", ""), ("²", ""), ("³", ""), ("⁴", ""),
    ("⁵", ""), ("⁶", ""), ("⁷", ""), ("⁸", ""), ("⁹", ""),
    # ASCII digits: unspoken markers glued to words (e.g. ਪਉੜੀ੫॥, ਗਉੜੀ੯॥) -> drop
    ("0", ""), ("1", ""), ("2", ""), ("3", ""), ("4", ""),
    ("5", ""), ("6", ""), ("7", ""), ("8", ""), ("9", ""),
]


def ctc_form(nw: str) -> str:
    t = nw
    for d, p in NUKTA:
        t = t.replace(d, p)
    # chars missing from the wav2vec2 Punjabi model vocab, mapped to the
    # closest available char (found by scanning SGGS text 2026-09-26)
    for d, p in VOCAB_FIXUPS:
        t = t.replace(d, p)
    return t


def resolve_device(prefer=None):
    import torch
    if prefer in ("cuda", "mps", "cpu"):
        return prefer
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def compute_emissions(model, processor, wav, device):
    import torch
    w = wav - wav.mean()
    std = w.std()
    if std > 1e-7:
        w = w / std
    CHUNK_S, OVERLAP_S = 90, 2
    sr = 16000
    n = len(w)
    cs, ov = int(CHUNK_S * sr), int(OVERLAP_S * sr)
    of = int(OVERLAP_S / FRAME_S)
    parts = []
    pos = 0
    while pos < n:
        s = max(0, pos - (ov if pos > 0 else 0))
        e = min(n, pos + cs + (ov if pos + cs < n else 0))
        inputs = processor(w[s:e], sampling_rate=sr, return_tensors="pt")
        with torch.no_grad():
            logits = model(inputs.input_values.to(device)).logits
        em_c = torch.log_softmax(logits, dim=-1).float().cpu().numpy()[0]
        f_s = 0 if pos == 0 else of
        f_e = em_c.shape[0] if e == n else em_c.shape[0] - of
        parts.append(em_c[f_s:f_e].astype(np.float32))
        pos += cs
    return np.concatenate(parts, axis=0)


def srt_timestamp(sec):
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    ms = int(round((sec - int(sec)) * 1000))
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def write_srt(line_cues, lines, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, ((s, e), line) in enumerate(zip(line_cues, lines), 1):
            f.write(f"{i}\n{srt_timestamp(s)} --> {srt_timestamp(e)}\n"
                    f"{line}\n\n")


def align_ang(ang, text_path, audio_path, out_prefix, device="auto",
              verbose=True):
    """Full pipeline for one Ang. Returns the diagnostics dict."""
    import torch
    import librosa
    from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
    from ctc_forced_aligner import forced_align, merge_repeats

    def log(msg):
        if verbose:
            print(msg, flush=True)

    device = resolve_device(None if device == "auto" else device)
    log(f"[ang {ang}] device: {device}")

    log("  loading model ...")
    t0 = time.time()
    processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
    model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(device).eval()
    vocab = processor.tokenizer.get_vocab()
    log(f"  model loaded in {time.time() - t0:.0f}s")

    wav, _ = librosa.load(audio_path, sr=16000, mono=True)
    log(f"  {len(wav) / 16000:.0f}s audio")

    em_cache = out_prefix + ".emissions.npz"
    t0 = time.time()
    if os.path.exists(em_cache):
        z = np.load(em_cache)
        if int(z["n_samples"]) != len(wav):
            raise RuntimeError(f"stale cache {em_cache}; delete it")
        emissions = z["emissions"].astype(np.float32)
        log(f"  cached emissions in {time.time() - t0:.0f}s")
    else:
        emissions = compute_emissions(model, processor, wav, device)
        log(f"  emissions computed in {time.time() - t0:.0f}s")
        np.savez(em_cache, emissions=emissions,
                 n_samples=np.int64(len(wav)))
    n_frames = emissions.shape[0]

    lines = load_lines(text_path)
    ref = []
    for li, line in enumerate(lines):
        for tok in line.split():
            nw = norm(tok)
            if nw:
                ref.append((tok, nw, li))
    ref_line = [li for (_, _, li) in ref]
    log(f"  {len(ref)} canonical words, {len(lines)} lines")

    tgt_ids, char_word, valid = [], [], []
    for i, (tok, nw, _li) in enumerate(ref):
        if DIGIT_RE.fullmatch(nw):
            valid.append(False)
            continue
        pieces = IK_OANKAR_EXPANSION if nw == "ੴ" else [ctc_form(nw)]
        for piece in pieces:
            if tgt_ids:
                tgt_ids.append(SEP_ID)
                char_word.append(None)
            for ch in piece:
                cid = vocab.get(ch)
                if cid is None:
                    raise RuntimeError(f"char {ch!r} not in model vocab")
                tgt_ids.append(cid)
                char_word.append(i)
        valid.append(True)

    targets = np.asarray([tgt_ids], dtype=np.int64)
    path, _scores = forced_align(np.expand_dims(emissions, 0), targets,
                                 blank=BLANK_ID)
    path = path.squeeze()
    idx_to_token = {v: k for k, v in vocab.items()}
    idx_to_token[BLANK_ID] = "<blank>"
    segments = merge_repeats(path.tolist(), idx_to_token)

    exp_chars = [(char_word[pos], idx_to_token[cid])
                 for pos, cid in enumerate(tgt_ids) if cid != SEP_ID]
    n_sep_expected = sum(1 for cid in tgt_ids if cid == SEP_ID)
    word_frames, n_sep_seen, n_skipped = {}, 0, 0
    skipped_words = set()
    last_matched_label = None
    ei = 0
    for si, seg in enumerate(segments):
        if ei >= len(exp_chars):
            break
        if seg.label == "<blank>":
            continue
        if seg.label == "|":
            n_sep_seen += 1
            continue
        L = seg.label
        # The Viterbi path may emit the same label in several
        # blank-separated runs (e.g. h, blank, h for target "hh"); each run
        # aligns to ONE target char. Only a single merged run with no
        # same-label run following it may cover a whole run of identical
        # target chars.
        run_ahead = 0
        j = si + 1
        while j < len(segments):
            lj = segments[j].label
            if lj == "<blank>":
                j += 1
                continue
            if lj == L:
                run_ahead += 1
                j += 1
                continue
            break
        start = ei
        while (start < len(exp_chars) and start - ei <= MAX_SKIP
               and exp_chars[start][1] != L):
            start += 1
        if start >= len(exp_chars) or exp_chars[start][1] != L:
            # model stutter: re-emitted the just-matched label with no
            # target char left for it -> ignore the extra segment
            if last_matched_label == L:
                continue
            raise RuntimeError(f"walk desync at segment {seg}")
        for k in range(ei, start):
            n_skipped += 1
            skipped_words.add(exp_chars[k][0])
        ei = start
        wi_seg = None
        while ei < len(exp_chars) and exp_chars[ei][1] == L:
            wi, _ch = exp_chars[ei]
            if wi_seg is None:
                wi_seg = wi
            elif wi != wi_seg:
                break
            if wi not in word_frames:
                word_frames[wi] = [seg.start, seg.end]
            else:
                word_frames[wi][1] = seg.end
            last_matched_label = L
            ei += 1
            if run_ahead > 0:
                break

    n_exp = len(exp_chars)
    # ei already jumps over skipped chars, so it counts every expected char
    # exactly once; n_skipped is tracked separately for diagnostics
    assert ei == n_exp, \
        f"walk consumed {ei}/{n_exp} chars ({n_skipped} skipped)"
    assert n_sep_seen <= n_sep_expected
    prev_end = -1
    for i in sorted(word_frames):
        s, e = word_frames[i]
        assert 0 <= s <= e < n_frames
        assert s > prev_end or i == 0
        prev_end = e

    hyp_words, scores, agreements = [], [], []
    argmax = emissions.argmax(axis=-1)
    for i, (tok, nw, _li) in enumerate(ref):
        if valid[i] and i in word_frames:
            s, e = word_frames[i]
            st, en = round(s * FRAME_S, 2), round((e + 1) * FRAME_S, 2)
            fr = path[s:e + 1]
            sc = float(emissions[s:e + 1, fr].mean())
            char_mask = fr != BLANK_ID
            agr = (float((argmax[s:e + 1][char_mask] == fr[char_mask]).mean())
                   if char_mask.sum() else float("nan"))
            agreements.append(agr)
            hyp_words.append({"word": tok, "start": st, "end": en})
            scores.append(sc)
        else:
            hyp_words.append({"word": tok, "start": 0.0, "end": 0.0})
            scores.append(float("nan"))
            agreements.append(float("nan"))

    n_direct = sum(1 for i in range(len(ref)) if valid[i] and i in word_frames)
    n_interp = len(ref) - n_direct
    sc_arr = np.array([s for s in scores if not np.isnan(s)])
    mean_sc = float(sc_arr.mean()) if len(sc_arr) else float("nan")
    ag_arr = np.array([a for a in agreements if not np.isnan(a)])
    mean_ag = float(ag_arr.mean()) if len(ag_arr) else float("nan")

    pairs = [(i, i) for i in range(len(ref))
             if valid[i] and i in word_frames]
    audio_dur = n_frames * FRAME_S
    line_cues, hits = build_cues(lines, ref_line, hyp_words, pairs, audio_dur)
    word_timings = build_word_timings(lines, ref, hyp_words, pairs, line_cues)

    diagnostics = {
        "ref_words": len(ref),
        "words_direct_timing": n_direct,
        "words_interpolated": n_interp,
        "anchored_lines": len(hits),
        "total_lines": len(lines),
        "mean_word_logprob": round(mean_sc, 3),
        "mean_argmax_agreement": round(mean_ag, 3),
        "chars_skipped_by_path": n_skipped,
        "words_with_skipped_chars": len(skipped_words),
    }
    payload = {
        "audio": audio_path,
        "text": text_path,
        "duration": round(audio_dur, 2),
        "model": f"{MODEL_ID} (CTC)",
        "lines": [
            {"line": num, "start": round(s, 2), "end": round(e, 2),
             "text": line, "words": wt}
            for num, ((s, e), line, wt) in
            enumerate(zip(line_cues, lines, word_timings), 1)
        ],
        "diagnostics": diagnostics,
    }
    json_path = out_prefix + ".json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=1)
    write_srt(line_cues, lines, out_prefix + ".srt")
    log(f"  wrote {json_path} + .srt "
        f"({n_direct}/{len(ref)} direct, agreement {mean_ag:.3f})")
    return diagnostics


def align_one(ang, out_dir="output", device="auto"):
    """Fetch (if needed) + align one Ang. Picklable for worker pools."""
    txt, mp3 = fetch_ang(ang,
                         text_dir=os.path.join(out_dir, "text"),
                         audio_dir=os.path.join(out_dir, "audio"))
    os.makedirs(out_dir, exist_ok=True)
    return align_ang(ang, txt, mp3, os.path.join(out_dir, f"ang{ang}.ctc"),
                     device=device, verbose=False)


def align_one_star(args):
    """Star-args wrapper so Pool.map can be used with one tuple."""
    return align_one(*args)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("start", type=int)
    ap.add_argument("end", type=int, nargs="?")
    ap.add_argument("--out", default="output")
    ap.add_argument("--device", default="auto")
    args = ap.parse_args()
    end = args.end or args.start
    for ang in range(args.start, end + 1):
        d = align_one(ang, out_dir=args.out, device=args.device)
        print(f"ang{ang}: {d['words_direct_timing']}/{d['ref_words']} "
              f"direct, agreement {d['mean_argmax_agreement']:.3f}")


if __name__ == "__main__":
    main()


In [ ]:
import os
import gurbani_worker as W

# ---------------- config ----------------
NOTEBOOK_VERSION = 2  # must match the generator; uploads are versioned v1, v2, ...
ANG_START = 12        # first Ang (1-1430)
ANG_END   = 21        # last Ang (inclusive)
WORKERS   = 2         # parallel Angs: 2 is safe on Mac 8GB / Colab T4 / Kaggle P100; raise on bigger machines
OUT_DIR   = "gurbani_output"
# ----------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
print("notebook version:", NOTEBOOK_VERSION)
print("device :", W.resolve_device())
print("range  :", ANG_START, "->", ANG_END, f"({ANG_END - ANG_START + 1} angs)")
print("workers:", WORKERS)

In [ ]:
import glob
import os
import zipfile
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import gurbani_worker as W

angs = list(range(ANG_START, ANG_END + 1))

# fetch text+audio first (fast, sequential, with retries)
for a in angs:
    W.fetch_ang(a, text_dir=os.path.join(OUT_DIR, "text"),
                   audio_dir=os.path.join(OUT_DIR, "audio"))
print(f"fetched {len(angs)} angs")

# download the model ONCE so parallel workers share the HF cache
# instead of each trying to fetch 1.2GB at the same time
from huggingface_hub import snapshot_download
print("caching model ...")
snapshot_download(repo_id=W.MODEL_ID)
print("model cached")


def write_checkpoint(n_ok):
    # cumulative checkpoint ZIP of everything aligned so far
    zpath = f"gurbani_batch_checkpoint_{n_ok}.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(glob.glob(os.path.join(OUT_DIR, "ang*.ctc.json")) +
                        glob.glob(os.path.join(OUT_DIR, "ang*.ctc.srt"))):
            z.write(p)
    print(f"\ncheckpoint: {zpath} ({os.path.getsize(zpath)//1024} KB) "
          f"- {n_ok} angs safe")


CKPT_EVERY = 35  # cumulative checkpoint every this many successful Angs

# align in parallel; each worker loads its own model copy
import multiprocessing as mp
# forked workers cannot re-initialize CUDA after the parent touched it
# (resolve_device in the config cell calls torch.cuda.is_available());
# spawn gives each worker a fresh process with its own CUDA context
mp.set_start_method("spawn", force=True)
jobs = [(a, OUT_DIR, "auto") for a in angs]
results = {}
last_ckpt = 0
with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(W.align_one_star, j): j[0] for j in jobs}
    for f in tqdm(futs, total=len(futs), desc="aligning"):
        a = futs[f]
        try:
            results[a] = f.result()
        except Exception as e:
            results[a] = {"error": str(e)}
            print(f"\nang{a} FAILED: {e}")
        n_ok = sum(1 for r in results.values() if "error" not in r)
        if n_ok - last_ckpt >= CKPT_EVERY:
            last_ckpt = n_ok
            write_checkpoint(n_ok)

n_ok = sum(1 for r in results.values() if "error" not in r)
print(f"\ndone: {n_ok}/{len(angs)} succeeded")

In [ ]:
import glob, json, os, zipfile

rows = []
for a in sorted(results):
    r = results[a]
    if "error" in r:
        rows.append((a, "FAILED", r["error"][:60], "", ""))
    else:
        rows.append((a, f'{r["words_direct_timing"]}/{r["ref_words"]}',
                     f'{r["mean_argmax_agreement"]:.3f}',
                     f'{r["anchored_lines"]}/{r["total_lines"]}',
                     f'{r["words_interpolated"]}'))

print(f'{"ang":>5} {"direct":>9} {"agree":>6} {"lines":>7} {"interp":>6}')
for a, d, ag, ln, it in rows:
    print(f"{a:>5} {d:>9} {ag:>6} {ln:>7} {it:>6}")

zpath = "gurbani_batch.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(os.path.join(OUT_DIR, "ang*.ctc.json")) +
                    glob.glob(os.path.join(OUT_DIR, "ang*.ctc.srt"))):
        z.write(p)
print("\nzip:", zpath, f"({os.path.getsize(zpath)//1024} KB)")

In [ ]:
import glob, os
zip_abs = os.path.abspath("gurbani_batch.zip")
ckpts = sorted(glob.glob("gurbani_batch_checkpoint_*.zip"))
if ckpts:
    print("checkpoints on disk:",
          ", ".join(os.path.basename(c) for c in ckpts))
try:
    from google.colab import files
    files.download("gurbani_batch.zip")
    print("download started in Colab")
except ImportError:
    if os.path.isdir("/kaggle/working"):
        print("on Kaggle - download from the output file viewer (right sidebar):", zip_abs)
    else:
        print("not on Colab - your files are in:", zip_abs)
print("if the run died early, grab the latest gurbani_batch_checkpoint_N.zip instead")